# Codex Python SDK Walkthrough

This notebook demonstrates the core public API (`Codex`, `Thread`, `Turn`, `TurnResult`) and common usage patterns.

In [ ]:
from codex_app_server import Codex, TextInput
from codex_app_server.retry import retry_on_overload
from codex_app_server.errors import JsonRpcError

In [ ]:
with Codex() as codex:
    print('server_name:', codex.metadata.server_name)
    print('server_version:', codex.metadata.server_version)

    thread = codex.thread_start(model='gpt-5')
    result = retry_on_overload(lambda: thread.turn(TextInput('Reply with exactly NOTEBOOK_OK')).run())

    print('thread_id:', result.thread_id)
    print('turn_id:', result.turn_id)
    print('status:', result.status)
    print('text:', (result.text or '').strip())

## Continue the same thread

Calling `thread.turn(...)` repeatedly preserves context.

In [ ]:
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5')
    _ = thread.turn(TextInput('Remember the codeword: ORBIT')).run()
    second = thread.turn(TextInput('What was the codeword?')).run()
    print((second.text or '').strip())

## Error handling pattern

In [ ]:
try:
    with Codex() as codex:
        thread = codex.thread_start(model='gpt-5')
        res = thread.turn(TextInput('Give me 2 bullets about retries')).run()
        print((res.text or '').strip())
except JsonRpcError as exc:
    print(f'RPC error {exc.code}: {exc.message}')